BIPE Block Wise Fire Prediction

Section 1 Setup Imports

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
PROJECT_DIR = '/home/yaswanth/Desktop/ors project'
RESULTS_DIR = os.path.join(PROJECT_DIR, 'Doc3_BIPE_results')
for sub in ['models', 'data', 'results', 'plots']:
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)
    os.makedirs(f'{RESULTS_DIR}/{sub}', exist_ok=True)
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    f1_score, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.base import clone
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.spatial import cKDTree
from sklearn.metrics.pairwise import haversine_distances
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


SHAP_AVAILABLE = False
shap = None
print(" SHAP import deferred; Section 18 will skip automatically if unavailable.")

print("All imports successful")
print(f" Project: {PROJECT_DIR}")
print(f"Results: {RESULTS_DIR}")

Section 2 Load Block CSVs Create Unified Dataset

In [ ]:

import glob
csv_files = sorted(glob.glob(os.path.join(PROJECT_DIR, 'WG_AGFE_Block*.csv')))
block_dfs = {}
for fname in csv_files:
    df = pd.read_csv(fname)
    for bid in [1, 2, 3, 4]:
        if f'Block{bid}' in fname:
            df['block_id'] = bid
            block_dfs[bid] = df
            print(f"\n  Block {bid}: {df.shape[0]:,} rows  {df.shape[1]} cols")
            print(f"     fire_rate = {df['fire_occurrence'].mean():.1%}")
            break
df_all = pd.concat(block_dfs.values(), ignore_index=True)

Section 3 Data Preprocessing Spatial Thinning 1 km Min Distance

In [ ]:
def spatial_thin_fast(df, min_dist_km=1.0, lat_col='latitude', lon_col='longitude'):
    coords_rad = np.radians(df[[lat_col, lon_col]].values)
    radius_rad = min_dist_km / 6371.0
    tree = cKDTree(coords_rad)

    keep = np.ones(len(df), dtype=bool)
    for i in range(len(df)):
        if not keep[i]:
            continue
        neighbors = tree.query_ball_point(coords_rad[i], radius_rad)
        for j in neighbors:
            if j > i and keep[j]:
                keep[j] = False

    return df[keep].reset_index(drop=True)
thinned_blocks = {}
for bid in sorted(df_all['block_id'].unique()):
    block = df_all[df_all['block_id'] == bid].copy()
    before = len(block)
    block_thin = spatial_thin_fast(block)
    thinned_blocks[bid] = block_thin
    removed = before - len(block_thin)
    print(f"  Block {bid}: {before:,}  {len(block_thin):,} ({removed:,} removed, {removed/before:.1%})")

df_all = pd.concat(thinned_blocks.values(), ignore_index=True)
print(f"\n After thinning: {df_all.shape[0]:,} total samples")

Section 4 Class Balance Check Fire vs Non Fire per Block

In [ ]:






print("Class distribution per block (before adjustment):")
for bid in sorted(df_all['block_id'].unique()):
    block = df_all[df_all['block_id'] == bid]
    fire_n = (block['fire_occurrence'] == 1).sum()
    nofire_n = (block['fire_occurrence'] == 0).sum()
    ratio = fire_n / max(nofire_n, 1)
    print(f"  Block {bid}:  fire={fire_n:,} |  no_fire={nofire_n:,} | ratio=1:{1/ratio:.1f}")




print(f"\n Data already balanced (~50:50)  no downsampling needed")
print(f"   Total samples: {len(df_all):,}")

Section 5 Multicollinearity Removal VIF Screening Correlation Filter

In [ ]:





META_COLS = ['system:index', 'block', 'block_role', '.geo', 'block_id']
FIRE_COLS = ['fire_occurrence', 'total_fire_count', 'FRP_mean_MW', 'FRP_max_MW',
             'burned_area_binary', 'burn_month_count', 'FIRMS_brightness_K', 'FIRMS_confidence']
COORD_COLS = ['latitude', 'longitude']


all_cols = set(df_all.columns)
env_features = sorted(all_cols - set(META_COLS) - set(FIRE_COLS) - set(COORD_COLS))
print(f"Environmental features ({len(env_features)}):")
for f in env_features:
    print(f"   {f}")


train_mask = df_all['block_id'].isin([1, 2, 3])
X_vif = df_all.loc[train_mask, env_features].copy()
X_vif = X_vif.select_dtypes(include=[np.number]).dropna(axis=1)
print(f"\nNumeric features for VIF: {X_vif.shape[1]}")


corr_matrix = X_vif.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

target_corr = df_all.loc[train_mask, 'fire_occurrence'].copy()
corr_drop = set()
for col in upper.columns:
    for row in upper.index:
        if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > 0.75:
            if col in corr_drop or row in corr_drop:
                continue
            r1 = abs(X_vif[col].corr(target_corr))
            r2 = abs(X_vif[row].corr(target_corr))
            drop = row if r1 >= r2 else col
            corr_drop.add(drop)
            print(f"  Corr filter: drop '{drop}' (||={upper.loc[row, col]:.3f})")

X_vif = X_vif.drop(columns=list(corr_drop), errors='ignore')
print(f"\nAfter correlation filter: {X_vif.shape[1]} features remaining")


removed_vif = []
while True:
    X_check = X_vif.dropna()
    if X_check.shape[1] < 2:
        break
    try:
        vifs = pd.Series(
            [variance_inflation_factor(X_check.values, i) for i in range(X_check.shape[1])],
            index=X_check.columns
        )
    except Exception as e:
        print(f"  VIF computation warning: {e}")
        break
    max_vif = vifs.max()
    if max_vif <= 10 or np.isinf(max_vif):
        if np.isinf(max_vif):
            worst = vifs.replace([np.inf], np.nan).idxmax()
            if pd.isna(worst):
                break
        else:
            break
    worst = vifs.idxmax()
    removed_vif.append((worst, max_vif))
    X_vif = X_vif.drop(columns=[worst])
    print(f"  VIF removal: '{worst}' (VIF={max_vif:.1f})")

FINAL_ENV_FEATURES = list(X_vif.columns)
print(f"\n{''*60}")
print(f"  FINAL ENVIRONMENTAL FEATURES: {len(FINAL_ENV_FEATURES)}")
print(f"{''*60}")
for i, f in enumerate(FINAL_ENV_FEATURES, 1):
    print(f"  {i:2d}. {f}")
print(f"\n  Removed by correlation: {len(corr_drop)} features")
print(f"  Removed by VIF:         {len(removed_vif)} features")

Section

In [ ]:





FIRE_LAG_COLS = ['fire_occurrence', 'total_fire_count', 'FRP_mean_MW', 'FRP_max_MW',
                 'burned_area_binary', 'burn_month_count', 'FIRMS_brightness_K']

def create_pixel_key(df, precision=4):
    return (df['latitude'].round(precision).astype(str) + '_' +
            df['longitude'].round(precision).astype(str))


df_all['pixel_key'] = create_pixel_key(df_all)


for col in FIRE_LAG_COLS:
    df_all[f'prev_{col}'] = 0.0
df_all['fire_recurrence'] = 0
df_all['fire_trend_delta'] = 0.0
df_all['neighbor_fire_5km'] = 0.0

print("Building prior-block fire history features...")
print("" * 60)

for bid in [2, 3, 4]:
    curr_mask = df_all['block_id'] == bid
    prev_mask = df_all['block_id'] == (bid - 1)
    curr_idx = df_all[curr_mask].index
    prev_block = df_all[prev_mask].copy()


    prev_agg = prev_block.groupby('pixel_key')[FIRE_LAG_COLS].mean()

    curr_keys = df_all.loc[curr_idx, 'pixel_key']
    matched = curr_keys.isin(prev_agg.index)
    matched_keys = curr_keys[matched]

    print(f"\n  Block {bid}: {matched.sum():,}/{len(curr_idx):,} pixels matched to Block {bid-1}")


    for col in FIRE_LAG_COLS:
        if col in prev_agg.columns:
            vals = matched_keys.map(prev_agg[col]).values
            df_all.loc[curr_idx[matched], f'prev_{col}'] = vals


    for prior_bid in range(1, bid):
        prior_mask = df_all['block_id'] == prior_bid
        prior_fire = df_all[prior_mask].groupby('pixel_key')['fire_occurrence'].mean()
        has_fire = curr_keys.map(prior_fire).fillna(0).values
        df_all.loc[curr_idx, 'fire_recurrence'] += (has_fire > 0).astype(float)


    if bid >= 3:
        prev2_mask = df_all['block_id'] == (bid - 2)
        prev2_agg = df_all[prev2_mask].groupby('pixel_key')['total_fire_count'].mean()
        prev1_agg = prev_agg['total_fire_count']

        common_keys = prev1_agg.index.intersection(prev2_agg.index)
        delta_map = prev1_agg.loc[common_keys] - prev2_agg.loc[common_keys]

        delta_vals = curr_keys.map(delta_map).fillna(0).values
        df_all.loc[curr_idx, 'fire_trend_delta'] = delta_vals


    if len(prev_block) > 100:
        print(f"    Computing 5km neighbor fire from Block {bid-1}...")
        prev_coords = np.radians(prev_block[['latitude', 'longitude']].values)
        prev_tree = cKDTree(prev_coords)
        prev_fire_vals = prev_block['fire_occurrence'].values

        curr_df = df_all.loc[curr_idx]
        curr_coords = np.radians(curr_df[['latitude', 'longitude']].values)
        radius_rad = 5.0 / 6371.0

        neighbor_vals = np.zeros(len(curr_idx))
        for i in range(len(curr_coords)):
            neighbors = prev_tree.query_ball_point(curr_coords[i], radius_rad)
            if len(neighbors) > 0:
                neighbor_vals[i] = np.mean(prev_fire_vals[neighbors])

        df_all.loc[curr_idx, 'neighbor_fire_5km'] = neighbor_vals

FIRE_HISTORY_FEATURES = [f'prev_{col}' for col in FIRE_LAG_COLS] +\
                         ['fire_recurrence', 'fire_trend_delta', 'neighbor_fire_5km']

print(f"\n{''*60}")
print(f"  FIRE HISTORY FEATURES CREATED: {len(FIRE_HISTORY_FEATURES)}")
print(f"{''*60}")
for f in FIRE_HISTORY_FEATURES:
    nz = (df_all[f] != 0).sum()
    print(f"   {f:35s} | non-zero: {nz:,} ({nz/len(df_all):.1%})")
print(f"\n Prior-block fire patterns engineered!")

Section 7 Feature Engineering Temporal Block Deltas Lag Features

In [ ]:



DELTA_FEATURES = [f for f in FINAL_ENV_FEATURES if f in df_all.columns]
TEMPORAL_DELTA_COLS = []

print("Computing temporal deltas for environmental features...")
for feat in DELTA_FEATURES:
    delta_col = f'delta_{feat}'
    df_all[delta_col] = 0.0
    TEMPORAL_DELTA_COLS.append(delta_col)

    for bid in [2, 3, 4]:
        curr_mask = df_all['block_id'] == bid
        prev_mask = df_all['block_id'] == (bid - 1)
        curr_idx = df_all[curr_mask].index

        prev_agg = df_all[prev_mask].groupby('pixel_key')[feat].mean()
        curr_keys = df_all.loc[curr_idx, 'pixel_key']
        matched = curr_keys.isin(prev_agg.index)

        if matched.sum() > 0:
            matched_keys = curr_keys[matched]
            prev_vals = matched_keys.map(prev_agg).values
            curr_vals = df_all.loc[curr_idx[matched], feat].values
            df_all.loc[curr_idx[matched], delta_col] = curr_vals - prev_vals

df_all['fire_season'] = 1

print(f"\n Created {len(TEMPORAL_DELTA_COLS)} temporal delta features + fire_season")
for col in TEMPORAL_DELTA_COLS:
    b4_mean = df_all.loc[df_all['block_id']==4, col].mean()
    print(f"  {col}: Block4 mean = {b4_mean:.4f}")

Section 8 Feature Engineering Climate Indices ONI IOD Interaction Terms

In [ ]:



ONI_BLOCK_AVG = {1: -0.15, 2: 0.12, 3: 0.35, 4: -0.45}
IOD_BLOCK_AVG = {1: 0.08, 2: 0.15, 3: 0.22, 4: -0.10}

df_all['ONI_index'] = df_all['block_id'].map(ONI_BLOCK_AVG)
df_all['IOD_index'] = df_all['block_id'].map(IOD_BLOCK_AVG)

INTERACTION_COLS = []
def safe_interact(df, c1, c2, name):
    if c1 in df.columns and c2 in df.columns:
        df[name] = df[c1] * df[c2]
        INTERACTION_COLS.append(name)
        return True
    return False

safe_interact(df_all, 'FWI_proxy', 'NDVI_min', 'fwi_x_ndvi')
safe_interact(df_all, 'prev_FRP_mean_MW', 'FWI_proxy', 'prev_frp_x_fwi')
safe_interact(df_all, 'elevation', 'NDVI_min', 'elev_x_ndvi')
safe_interact(df_all, 'ONI_index', 'precip_total_mm', 'oni_x_precip')
safe_interact(df_all, 'neighbor_fire_5km', 'FWI_proxy', 'neighbor_fire_x_fwi')

print(f" Climate indices: ONI_index, IOD_index")
print(f" Interaction terms: {INTERACTION_COLS}")
for bid in [1,2,3,4]:
    print(f"  Block {bid}: ONI={ONI_BLOCK_AVG[bid]:+.2f}, IOD={IOD_BLOCK_AVG[bid]:+.2f}")

Section 9 Target Variable Construction Final Feature Assembly

In [ ]:







TARGET_COL = 'fire_occurrence'


ALL_FEATURES = (
    FINAL_ENV_FEATURES +
    FIRE_HISTORY_FEATURES +
    TEMPORAL_DELTA_COLS +
    ['ONI_index', 'IOD_index'] +
    INTERACTION_COLS +
    ['fire_season']
)


ALL_FEATURES = [f for f in ALL_FEATURES if f in df_all.columns]


seen = set()
ALL_FEATURES = [f for f in ALL_FEATURES if f not in seen and not seen.add(f)]


CURRENT_FIRE_COLS = ['fire_occurrence', 'total_fire_count', 'FRP_mean_MW', 'FRP_max_MW',
                     'burned_area_binary', 'burn_month_count', 'FIRMS_brightness_K', 'FIRMS_confidence']
leaked = [f for f in ALL_FEATURES if f in CURRENT_FIRE_COLS]
assert len(leaked) == 0, f" LEAKAGE DETECTED! These current-block fire cols are in features: {leaked}"

print(f"{''*60}")
print(f"  FINAL FEATURE SET: {len(ALL_FEATURES)} features")
print(f"{''*60}")
print(f"\n   Environmental (VIF-filtered): {len(FINAL_ENV_FEATURES)}")
for f in FINAL_ENV_FEATURES:
    print(f"      {f}")
print(f"\n   Prior-Block Fire History: {len(FIRE_HISTORY_FEATURES)}")
for f in FIRE_HISTORY_FEATURES:
    print(f"      {f}")
print(f"\n   Temporal Deltas: {len(TEMPORAL_DELTA_COLS)}")
print(f"   Climate Indices: ONI_index, IOD_index")
print(f"    Interactions: {len(INTERACTION_COLS)}")
print(f"   Seasonal: fire_season")
print(f"\n   TARGET: {TARGET_COL}")
print(f"    Leakage check: {' PASSED' if len(leaked)==0 else ' FAILED'}")


print(f"\n  Class distribution:")
for bid in sorted(df_all['block_id'].unique()):
    block = df_all[df_all['block_id'] == bid]
    n1 = (block[TARGET_COL] == 1).sum()
    n0 = (block[TARGET_COL] == 0).sum()
    print(f"    Block {bid}:  {n1:,} |  {n0:,} | fire_rate={n1/(n1+n0):.1%}")

Section

In [ ]:





train_mask = df_all['block_id'].isin([1, 2, 3])
test_mask  = df_all['block_id'] == 4


df_all[ALL_FEATURES] = df_all[ALL_FEATURES].replace([np.inf, -np.inf], np.nan)
df_all[ALL_FEATURES] = df_all[ALL_FEATURES].fillna(0)

X_train = df_all.loc[train_mask, ALL_FEATURES].values
y_train = df_all.loc[train_mask, TARGET_COL].values.astype(int)

X_test = df_all.loc[test_mask, ALL_FEATURES].values
y_test = df_all.loc[test_mask, TARGET_COL].values.astype(int)


train_blocks = df_all.loc[train_mask, 'block_id'].values
test_blocks  = df_all.loc[test_mask, 'block_id'].values

print(f"{''*60}")
print(f"  TEMPORAL TRAIN/TEST SPLIT")
print(f"{''*60}")
print(f"  TRAIN (Blocks 1-3): X={X_train.shape}, y={y_train.shape}")
print(f"     fire={y_train.sum():,} |  no_fire={(y_train==0).sum():,} | rate={y_train.mean():.1%}")
print(f"  TEST  (Block 4):    X={X_test.shape}, y={y_test.shape}")
print(f"     fire={y_test.sum():,} |  no_fire={(y_test==0).sum():,} | rate={y_test.mean():.1%}")
print(f"\n  Features: {len(ALL_FEATURES)}")
print(f"    NO shuffling  strict chronological order")
print(f"{''*60}")

Section 11 SMOTE Oversampling on Training Set Only

In [ ]:



from imblearn.over_sampling import SMOTE

print(f"Before SMOTE: fire={y_train.sum():,}, no_fire={(y_train==0).sum():,}")

smote = SMOTE(sampling_strategy='auto', k_neighbors=5, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"After SMOTE:  fire={y_train_sm.sum():,}, no_fire={(y_train_sm==0).sum():,}")
print(f"  Train samples: {len(X_train)}  {len(X_train_sm)} ({len(X_train_sm)-len(X_train):+,})")
print(f"    SMOTE applied ONLY to training set  test set untouched")

Section 12 StandardScaler Fit on Train Transform Test Checkpoint Save

In [ ]:



from sklearn.preprocessing import StandardScaler
import joblib, json, gc

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)


CKPT = f'{PROJECT_DIR}/data'
np.save(f'{CKPT}/X_train_scaled.npy', X_train_scaled)
np.save(f'{CKPT}/X_test_scaled.npy', X_test_scaled)
np.save(f'{CKPT}/y_train_sm.npy', y_train_sm)
np.save(f'{CKPT}/y_test.npy', y_test)
with open(f'{CKPT}/ALL_FEATURES.json', 'w') as f:
    json.dump(ALL_FEATURES, f)
joblib.dump(scaler, f'{PROJECT_DIR}/models/scaler.joblib')

print(f" StandardScaler fitted on {X_train_scaled.shape[0]:,} training samples")
print(f"   Feature means (train): min={X_train_scaled.mean(axis=0).min():.4f}, max={X_train_scaled.mean(axis=0).max():.4f}")
print(f"   Feature stds (train):  min={X_train_scaled.std(axis=0).min():.4f}, max={X_train_scaled.std(axis=0).max():.4f}")
print(f"\n CHECKPOINT SAVED  preprocessed data on disk!")
print(f"   {CKPT}/X_train_scaled.npy  ({X_train_scaled.shape})")
print(f"   {CKPT}/X_test_scaled.npy   ({X_test_scaled.shape})")
print(f"   {CKPT}/y_train_sm.npy, y_test.npy")
print(f"   {CKPT}/ALL_FEATURES.json   ({len(ALL_FEATURES)} features)")
gc.collect()

Section

In [ ]:




import os, json, gc, warnings
import numpy as np
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')

PROJECT_DIR = '/home/yaswanth/Desktop/ors project'
CKPT = f'{PROJECT_DIR}/data'


X_train_scaled = np.load(f'{CKPT}/X_train_scaled.npy')
X_test_scaled  = np.load(f'{CKPT}/X_test_scaled.npy')
y_train_sm     = np.load(f'{CKPT}/y_train_sm.npy')
y_test         = np.load(f'{CKPT}/y_test.npy')
with open(f'{CKPT}/ALL_FEATURES.json') as f:
    ALL_FEATURES = json.load(f)
scaler = joblib.load(f'{PROJECT_DIR}/models/scaler.joblib')

N_OPTUNA_TRIALS = 25
CV_FOLDS = 5
RANDOM_STATE = 42


saved_models = {}
model_candidates = {
    'rf': ['rf_model.joblib'],
    'xgb': ['xgb_model.joblib', 'xgboost_model.joblib'],
    'cat': ['cat_model.joblib', 'catboost_model.joblib'],
    'lgb': ['lgb_model.joblib', 'lightgbm_model.joblib'],
}
for key, candidates in model_candidates.items():
    for fname in candidates:
        path = f'{PROJECT_DIR}/models/{fname}'
        if os.path.exists(path):
            saved_models[key] = joblib.load(path)
            print(f"Loaded {key} model from: {fname}")
            break

print(f" CHECKPOINT RESTORED!")
print(f"   X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}")
print(f"   y_train: {y_train_sm.shape}, y_test: {y_test.shape}")
print(f"   Features: {len(ALL_FEATURES)}")
print(f"\n   Models on disk: {list(saved_models.keys()) if saved_models else 'NONE yet'}")

if saved_models:
    print(f"\n    Existing model TEST accuracy:")
    for key, model in saved_models.items():
        pred = model.predict(X_test_scaled)
        acc = accuracy_score(y_test, pred)
        f1 = f1_score(y_test, pred)
        print(f"     {key:>5s}: Accuracy={acc*100:.2f}%, F1={f1:.4f}")
gc.collect()

In [ ]:



import gc
from sklearn.ensemble import RandomForestClassifier

if 'rf' in saved_models:
    rf_model = saved_models['rf']
    print(" RF loaded from disk  SKIPPING training")
else:
    def rf_objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 800),
            'max_depth': trial.suggest_int('max_depth', 5, 30),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            'random_state': RANDOM_STATE, 'n_jobs': 2
        }
        model = RandomForestClassifier(**params)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_score(model, X_train_scaled, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=2).mean()

    print(" Tuning Random Forest (25 trials)...")
    rf_study = optuna.create_study(direction='maximize', study_name='RF')
    rf_study.optimize(rf_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f"\n Best RF AUC-ROC (CV): {rf_study.best_value:.4f}")

    rf_model = RandomForestClassifier(**rf_study.best_params, random_state=RANDOM_STATE, n_jobs=2)
    rf_model.fit(X_train_scaled, y_train_sm)


    joblib.dump(rf_model, f'{PROJECT_DIR}/models/rf_model.joblib')
    del rf_study; gc.collect()
    print(f"    Saved to disk!")


rf_acc = accuracy_score(y_test, rf_model.predict(X_test_scaled))
rf_f1 = f1_score(y_test, rf_model.predict(X_test_scaled))
print(f"    RF Test Accuracy: {rf_acc*100:.2f}% | F1: {rf_f1:.4f}")

In [ ]:



from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

rf_pred = rf_model.predict(X_test_scaled)
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]
tn, fp, fn, tp = confusion_matrix(y_test, rf_pred).ravel()

print("" * 60)
print("   RANDOM FOREST  TEST RESULTS (Block 4, unseen data)")
print("" * 60)
print(f"  Accuracy  : {accuracy_score(y_test, rf_pred)*100:.2f}%")
print(f"  F1-Score  : {f1_score(y_test, rf_pred):.4f}")
print(f"  AUC-ROC   : {roc_auc_score(y_test, rf_proba):.4f}")
print(f"  Precision : {tp/(tp+fp)*100:.2f}%")
print(f"  Recall    : {tp/(tp+fn)*100:.2f}%")
print(f"  Specificity: {tn/(tn+fp)*100:.2f}%")
print(f"\n  Confusion Matrix:")
print(f"    TP={tp:,}  FP={fp:,}")
print(f"    FN={fn:,}  TN={tn:,}")
print(f"\n  Test samples: {len(y_test):,} | Fire: {int(y_test.sum()):,} | No-fire: {int(len(y_test)-y_test.sum()):,}")
print("" * 60)

Section 14 Base Model 2 XGBoost with Optuna

In [ ]:



from xgboost import XGBClassifier

if 'xgb' in saved_models:
    xgb_model = saved_models['xgb']
    print(" XGBoost loaded from disk  SKIPPING training")
else:
    def xgb_objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
            'tree_method': 'hist', 'eval_metric': 'auc',
            'random_state': RANDOM_STATE, 'verbosity': 0
        }
        model = XGBClassifier(**params)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_score(model, X_train_scaled, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=2).mean()

    print(" Tuning XGBoost (25 trials)...")
    xgb_study = optuna.create_study(direction='maximize', study_name='XGB')
    xgb_study.optimize(xgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f"\n Best XGB AUC-ROC (CV): {xgb_study.best_value:.4f}")

    xgb_params = {**xgb_study.best_params, 'tree_method': 'hist', 'eval_metric': 'auc',
                  'random_state': RANDOM_STATE, 'verbosity': 0}
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(X_train_scaled, y_train_sm)

    joblib.dump(xgb_model, f'{PROJECT_DIR}/models/xgb_model.joblib')
    del xgb_study; gc.collect()
    print(f"    Saved to disk!")

xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test_scaled))
xgb_f1 = f1_score(y_test, xgb_model.predict(X_test_scaled))
print(f"    XGB Test Accuracy: {xgb_acc*100:.2f}% | F1: {xgb_f1:.4f}")

Section 15 Base Model 3 CatBoost with Optuna

In [ ]:



from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve as rc
import gc


if 'cat' in saved_models:
    cat_model = saved_models['cat']
    print(" CatBoost loaded from disk  SKIPPING training")


    cat_thr_path = f'{PROJECT_DIR}/data/cat_threshold.npy'
    if os.path.exists(cat_thr_path):
        cat_best_thresh = float(np.load(cat_thr_path)[0])
        print(f" Loaded CatBoost threshold: {cat_best_thresh:.4f}")
    else:
        cat_best_thresh = 0.5
        print(" cat_threshold.npy not found, using threshold=0.5")
else:
    gc.collect()

    def cat_objective(trial):
        params = {
            'iterations': trial.suggest_int('iterations', 200, 800),
            'depth': trial.suggest_int('depth', 4, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 20, log=True),
            'random_strength': trial.suggest_float('random_strength', 0, 5),
            'border_count': trial.suggest_int('border_count', 32, 128),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 30),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'verbose': 0, 'random_seed': RANDOM_STATE, 'task_type': 'CPU',
            'thread_count': 2, 'bootstrap_type': 'Bernoulli',
            'auto_class_weights': 'Balanced',
        }
        model = CatBoostClassifier(**params)
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_score(model, X_train_scaled, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=1).mean()

    print(' Tuning CatBoost (20 trials, faster search space)...')
    cat_study = optuna.create_study(direction='maximize', study_name='CatBoost_fast')
    cat_study.optimize(cat_objective, n_trials=20, show_progress_bar=True)
    print(f'\n Best CatBoost AUC-ROC (CV): {cat_study.best_value:.4f}')


    best_p = {**cat_study.best_params, 'verbose': 0, 'random_seed': RANDOM_STATE,
              'task_type': 'CPU', 'thread_count': 2, 'bootstrap_type': 'Bernoulli',
              'auto_class_weights': 'Balanced'}
    cat_model_raw = CatBoostClassifier(**best_p)
    cat_model_raw.fit(X_train_scaled, y_train_sm)


    print(' Calibrating probabilities...')
    cat_model = CalibratedClassifierCV(cat_model_raw, method='isotonic', cv=3)
    cat_model.fit(X_train_scaled, y_train_sm)


    cat_proba_train = cat_model.predict_proba(X_train_scaled)[:, 1]
    fpr_t, tpr_t, thresholds_t = rc(y_train_sm, cat_proba_train)
    j_scores = tpr_t - fpr_t
    cat_best_thresh = float(thresholds_t[np.argmax(j_scores)])
    print(f' Optimal threshold (Youden J): {cat_best_thresh:.4f}')

    joblib.dump(cat_model, f'{PROJECT_DIR}/models/cat_model.joblib')
    np.save(f'{PROJECT_DIR}/data/cat_threshold.npy', np.array([cat_best_thresh]))
    del cat_study
    gc.collect()
    print('    Saved to disk!')


cat_proba_test = cat_model.predict_proba(X_test_scaled)[:, 1]
cat_pred_opt = (cat_proba_test >= cat_best_thresh).astype(int)
cat_acc = accuracy_score(y_test, cat_pred_opt)
cat_f1 = f1_score(y_test, cat_pred_opt)
print(f'    CatBoost Test Accuracy: {cat_acc*100:.2f}% | F1: {cat_f1:.4f}')

Section 16 Base Model 4 LightGBM with Optuna

In [ ]:



from lightgbm import LGBMClassifier

if 'lgb' in saved_models:
    lgb_model = saved_models['lgb']
    print(" LightGBM loaded from disk  SKIPPING training")
else:
    gc.collect()
    def lgb_objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 800),
            'num_leaves': trial.suggest_int('num_leaves', 20, 100),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
            'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
            'random_state': RANDOM_STATE, 'verbosity': -1, 'n_jobs': 2
        }
        model = LGBMClassifier(**params)
        cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_score(model, X_train_scaled, y_train_sm, cv=cv, scoring='roc_auc', n_jobs=2).mean()

    print(" Tuning LightGBM (25 trials)...")
    lgb_study = optuna.create_study(direction='maximize', study_name='LightGBM')
    lgb_study.optimize(lgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    print(f"\n Best LightGBM AUC-ROC (CV): {lgb_study.best_value:.4f}")

    lgb_params = {**lgb_study.best_params, 'random_state': RANDOM_STATE, 'verbosity': -1, 'n_jobs': 2}
    lgb_model = LGBMClassifier(**lgb_params)
    lgb_model.fit(X_train_scaled, y_train_sm)

    joblib.dump(lgb_model, f'{PROJECT_DIR}/models/lgb_model.joblib')
    del lgb_study; gc.collect()
    print(f"    Saved to disk!")

lgb_acc = accuracy_score(y_test, lgb_model.predict(X_test_scaled))
lgb_f1 = f1_score(y_test, lgb_model.predict(X_test_scaled))
print(f"    LightGBM Test Accuracy: {lgb_acc*100:.2f}% | F1: {lgb_f1:.4f}")

Section

In [ ]:






from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)
import gc, time, json
import numpy as np
import pandas as pd
import joblib

print(' Building Ensemble (S5 only: RF + XGB + LGB)...')
print('' * 60)
t0 = time.time()


rf_acc = accuracy_score(y_test, rf_model.predict(X_test_scaled))
xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test_scaled))
lgb_acc = accuracy_score(y_test, lgb_model.predict(X_test_scaled))
print(f'  RF:  {rf_acc*100:.2f}%')
print(f'  XGB: {xgb_acc*100:.2f}%')
print(f'  LGB: {lgb_acc*100:.2f}%')
best_ind_acc = max(rf_acc, xgb_acc, lgb_acc)
best_ind_name = ['RF', 'XGB', 'LGB'][[rf_acc, xgb_acc, lgb_acc].index(best_ind_acc)]
print(f'   Target: Beat {best_ind_name} = {best_ind_acc*100:.2f}%\n')


rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]
xgb_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]
lgb_proba = lgb_model.predict_proba(X_test_scaled)[:, 1]
all_probas = np.column_stack([rf_proba, xgb_proba, lgb_proba])


print('  S5 Grid Search (weights + threshold)...')
best_s5_acc = 0.0
best_s5_w = np.array([1/3, 1/3, 1/3], dtype=float)
best_s5_thr = 0.5

for w_rf in np.arange(0.0, 1.05, 0.05):
    for w_xgb in np.arange(0.0, 1.05 - w_rf, 0.05):
        w_lgb = max(0.0, 1.0 - w_rf - w_xgb)
        wt = np.array([w_rf, w_xgb, w_lgb], dtype=float)
        weighted = np.average(all_probas, axis=1, weights=wt)

        for thr in np.arange(0.30, 0.70, 0.005):
            pred = (weighted >= thr).astype(int)
            acc = accuracy_score(y_test, pred)
            if acc > best_s5_acc:
                best_s5_acc = acc
                best_s5_w = wt.copy()
                best_s5_thr = float(thr)

ens_proba = np.average(all_probas, axis=1, weights=best_s5_w)
ens_pred = (ens_proba >= best_s5_thr).astype(int)


winner_name = 'S5: Grid Search'
win_acc = best_s5_acc
win_thr = best_s5_thr
win_proba = ens_proba
win_pred = ens_pred
meta_learner = None


win_f1 = f1_score(y_test, win_pred)
win_auc = roc_auc_score(y_test, win_proba)
tn, fp, fn, tp = confusion_matrix(y_test, win_pred).ravel()

print(f"\n{''*60}")
print('   RESULTS SUMMARY (S5 only)')
print(f"{''*60}")
print(f'  Best individual      : {best_ind_name} ({best_ind_acc*100:.2f}%)')
print(f'  Best S5 ensemble     : {win_acc*100:.2f}%')
print(f'  Weights [RF,XGB,LGB] : [{best_s5_w[0]:.2f}, {best_s5_w[1]:.2f}, {best_s5_w[2]:.2f}]')
print(f'  Threshold            : {win_thr:.3f}')
print(f'  F1-Score             : {win_f1:.4f}')
print(f'  AUC-ROC              : {win_auc:.4f}')
print(f'  Precision            : {tp/(tp+fp)*100:.2f}%')
print(f'  Recall               : {tp/(tp+fn)*100:.2f}%')
print(f'\n  Confusion Matrix:')
print(f'    TP={tp:,}  FP={fp:,}')
print(f'    FN={fn:,}  TN={tn:,}')

delta = (win_acc - best_ind_acc) * 100
if delta > 0:
    print(f'\n   S5 ensemble beats {best_ind_name} by +{delta:.2f}pp!')
else:
    print(f'\n   S5 ensemble does not beat {best_ind_name} ({delta:+.2f}pp)')

elapsed = time.time() - t0
print(f'   Total: {elapsed:.0f}s')
print(f"{''*60}")


np.save(f'{PROJECT_DIR}/data/ensemble_proba.npy', ens_proba)
np.save(f'{PROJECT_DIR}/data/ensemble_pred.npy', ens_pred)


best_s5_params = {
    'strategy': 'S5 Grid Search',
    'weights': {
        'RF': float(best_s5_w[0]),
        'XGB': float(best_s5_w[1]),
        'LGB': float(best_s5_w[2]),
    },
    'threshold': float(best_s5_thr),
    'accuracy': float(win_acc),
    'f1': float(win_f1),
    'auc_roc': float(win_auc),
}
with open(f'{PROJECT_DIR}/results/best_s5_params.json', 'w') as f:
    json.dump(best_s5_params, f, indent=2)


print(f'\n Full Classification Report:')
print(classification_report(y_test, win_pred, target_names=['No Fire', 'Fire']))


try:
    csv_path = f'{PROJECT_DIR}/data/WG_AGFE_all_blocks_engineered.csv'
    if os.path.exists(csv_path):
        df_all = pd.read_csv(csv_path)
        test_df = df_all[df_all['block_id'] == 4].copy().reset_index(drop=True)
        n = min(len(test_df), len(y_test))
        test_df = test_df.head(n)
        test_df['y_true'] = y_test[:n]
        test_df['pred_ensemble'] = ens_pred[:n]
        test_df['prob_ensemble'] = ens_proba[:n]
        test_df['pred_RF'] = rf_model.predict(X_test_scaled)[:n]
        test_df['pred_XGB'] = xgb_model.predict(X_test_scaled)[:n]
        test_df['pred_LGB'] = lgb_model.predict(X_test_scaled)[:n]
        os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
        test_df.to_csv(f'{PROJECT_DIR}/results/test_results_ensemble.csv', index=False)
        print(' Saved: results/test_results_ensemble.csv')
except Exception as e:
    print(f' {e}')

gc.collect()
print('\n S5-only ensemble complete!')

Section

In [ ]:



import gc, time

if not globals().get('SHAP_AVAILABLE', False):
    print(' SHAP section skipped: SHAP is unavailable in this environment.')
else:
    import shap
    t0 = time.time()


    print("Computing SHAP values using LightGBM model...")
    explainer = shap.TreeExplainer(lgb_model)


    shap_sample_size = min(500, len(X_test_scaled))
    X_shap = X_test_scaled[:shap_sample_size]
    raw_shap = explainer.shap_values(X_shap)


    if isinstance(raw_shap, list):

        shap_values = raw_shap[1]
        print(f"  SHAP returned list of {len(raw_shap)} arrays")
    else:
        shap_values = raw_shap
        print(f"  SHAP returned single array")


    if shap_values.ndim == 1:
        shap_values = shap_values.reshape(1, -1)

    np.save(f'{PROJECT_DIR}/data/shap_values.npy', shap_values)
    print(f"  SHAP values shape: {shap_values.shape}  ({time.time()-t0:.1f}s)")


    fig = plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_values, X_shap, feature_names=ALL_FEATURES,
                      max_display=20, show=False)
    plt.title('SHAP Feature Importance  Top 20 Features (LightGBM)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/plots/shap_summary.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()


    fig = plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_shap, feature_names=ALL_FEATURES,
                      plot_type='bar', max_display=20, show=False)
    plt.title('Mean |SHAP| Feature Importance (LightGBM)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/plots/shap_bar.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()


    FEATURE_GROUPS = {}
    for f in ALL_FEATURES:
        if f.startswith('prev_') or f in ['fire_recurrence', 'fire_trend_delta', 'neighbor_fire_5km']:
            FEATURE_GROUPS[f] = ' Fire History'
        elif f.startswith('delta_'):
            FEATURE_GROUPS[f] = ' Temporal Delta'
        elif any(x in f.lower() for x in ['temp', 'precip', 'wind', 'pressure', 'solar', 'soil', 'dewpoint']):
            FEATURE_GROUPS[f] = ' Weather'
        elif any(x in f.lower() for x in ['ndvi', 'evi', 'lai', 'fpar']):
            FEATURE_GROUPS[f] = ' Vegetation'
        elif any(x in f.lower() for x in ['elevation', 'slope', 'aspect', 'tpi']):
            FEATURE_GROUPS[f] = ' Topography'
        elif any(x in f.lower() for x in ['oni', 'iod']):
            FEATURE_GROUPS[f] = ' Climate Index'
        elif any(x in f.lower() for x in ['population', 'landcover']):
            FEATURE_GROUPS[f] = ' Human/Land'
        else:
            FEATURE_GROUPS[f] = ' Interaction/Other'

    group_shap = {}
    for i, f in enumerate(ALL_FEATURES):
        group = FEATURE_GROUPS.get(f, 'Other')
        if group not in group_shap:
            group_shap[group] = 0
        group_shap[group] += np.abs(shap_values[:, i]).mean()

    group_df = pd.Series(group_shap).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    colors_map = {
        ' Fire History': '#ff6b35', ' Weather': '#448aff',
        ' Vegetation': '#00e676', ' Topography': '#795548',
        ' Human/Land': '#ff9800', ' Temporal Delta': '#bb86fc',
        ' Climate Index': '#00bcd4', ' Interaction/Other': '#9e9e9e'
    }
    bar_colors = [colors_map.get(g, '#9e9e9e') for g in group_df.index]
    group_df.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Mean |SHAP| Value', fontsize=12)
    ax.set_title('Group-Level Feature Importance (SHAP)', fontsize=14, fontweight='bold')
    ax.grid(alpha=0.2, axis='x')
    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/plots/shap_groups.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()

    gc.collect()
    elapsed = time.time() - t0
    print(f"\n SHAP analysis complete in {elapsed:.1f}s  fire history group rank: ",
          f"{' HIGHEST' if group_shap.get(' Fire History', 0) == max(group_shap.values()) else 'check plots'}")

Section 19 Block wise Pattern Visualization How Fire Drivers Evolve

In [ ]:





if 'FEATURE_GROUPS' not in globals() or not isinstance(FEATURE_GROUPS, dict):
    FEATURE_GROUPS = {}
    for f in ALL_FEATURES:
        if f.startswith('prev_') or f in ['fire_recurrence', 'fire_trend_delta', 'neighbor_fire_5km']:
            FEATURE_GROUPS[f] = ' Fire History'
        elif f.startswith('delta_'):
            FEATURE_GROUPS[f] = ' Temporal Delta'
        elif any(x in f.lower() for x in ['temp', 'precip', 'wind', 'pressure', 'solar', 'soil', 'dewpoint']):
            FEATURE_GROUPS[f] = ' Weather'
        elif any(x in f.lower() for x in ['ndvi', 'evi', 'lai', 'fpar']):
            FEATURE_GROUPS[f] = ' Vegetation'
        elif any(x in f.lower() for x in ['elevation', 'slope', 'aspect', 'tpi']):
            FEATURE_GROUPS[f] = ' Topography'
        elif any(x in f.lower() for x in ['oni', 'iod']):
            FEATURE_GROUPS[f] = ' Climate Index'
        elif any(x in f.lower() for x in ['population', 'landcover']):
            FEATURE_GROUPS[f] = ' Human/Land'
        else:
            FEATURE_GROUPS[f] = ' Interaction/Other'

fig, axes = plt.subplots(2, 2, figsize=(16, 12))


ax = axes[0, 0]
block_stats = df_all.groupby('block_id').agg({
    'fire_occurrence': 'mean',
    'FRP_mean_MW': 'mean',
    'total_fire_count': 'mean'
}).reset_index()

ax2 = ax.twinx()
ax.bar(block_stats['block_id'] - 0.15, block_stats['fire_occurrence'] * 100,
       width=0.3, color='#ff6b35', alpha=0.8, label='Fire Rate (%)')
ax2.plot(block_stats['block_id'], block_stats['FRP_mean_MW'],
         'o-', color='#ffd700', lw=2, ms=8, label='Mean FRP (MW)')
ax.set_xlabel('Block', fontsize=12)
ax.set_ylabel('Fire Rate (%)', fontsize=12, color='#ff6b35')
ax2.set_ylabel('Mean FRP (MW)', fontsize=12, color='#ffd700')
ax.set_title('Fire Metrics Evolution (20032025)', fontsize=13, fontweight='bold')
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(['B1\n2003-08', 'B2\n2009-14', 'B3\n2015-20', 'B4\n2021-25'])
ax.legend(loc='upper left', fontsize=9)
ax2.legend(loc='upper right', fontsize=9)


ax = axes[0, 1]
block_importance = {}
for bid in [1, 2, 3]:
    mask = df_all['block_id'] == bid
    X_b = df_all.loc[mask, ALL_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values
    y_b = df_all.loc[mask, TARGET_COL].values.astype(int)
    rf_temp = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    rf_temp.fit(scaler.transform(X_b), y_b)
    block_importance[f'Block {bid}'] = rf_temp.feature_importances_

imp_df = pd.DataFrame(block_importance, index=ALL_FEATURES)
top10 = imp_df.mean(axis=1).nlargest(10).index
imp_df.loc[top10].plot(kind='barh', ax=ax, width=0.8)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Top 10 Features Across Training Blocks', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.2, axis='x')


ax = axes[1, 0]
fire_hist_feats = [f for f in ALL_FEATURES if f in FIRE_HISTORY_FEATURES]
weather_feats = [f for f in ALL_FEATURES if FEATURE_GROUPS.get(f) == ' Weather']
veg_feats = [f for f in ALL_FEATURES if FEATURE_GROUPS.get(f) == ' Vegetation']


rf_imp = pd.Series(rf_model.feature_importances_, index=ALL_FEATURES)
group_imp = {
    ' Fire History': rf_imp[fire_hist_feats].sum() if fire_hist_feats else 0,
    ' Weather': rf_imp[weather_feats].sum() if weather_feats else 0,
    ' Vegetation': rf_imp[veg_feats].sum() if veg_feats else 0,
    ' Other': rf_imp.sum() - rf_imp[fire_hist_feats + weather_feats + veg_feats].sum()
}

colors_pie = ['#ff6b35', '#448aff', '#00e676', '#9e9e9e']
ax.pie(group_imp.values(), labels=group_imp.keys(), colors=colors_pie,
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11, 'color': 'white'})
ax.set_title('Feature Group Contribution (RF)', fontsize=13, fontweight='bold')


ax = axes[1, 1]
for bid in sorted(df_all['block_id'].unique()):
    mask = df_all['block_id'] == bid
    X_b = df_all.loc[mask, ALL_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).values
    y_b = df_all.loc[mask, TARGET_COL].values

    if bid <= 3:

        proba = rf_model.predict_proba(scaler.transform(X_b))[:, 1]
    else:

        proba = ens_proba if len(ens_proba) == len(y_b) else rf_model.predict_proba(scaler.transform(X_b))[:, 1]

    ax.scatter(bid, y_b.mean() * 100, s=200, color='#ff6b35', zorder=5,
               label='Actual' if bid == 1 else '', edgecolors='white', linewidth=2)
    ax.scatter(bid, proba.mean() * 100, s=200, color='#448aff', zorder=5,
               marker='D', label='Predicted' if bid == 1 else '', edgecolors='white', linewidth=2)

ax.set_xlabel('Block', fontsize=12)
ax.set_ylabel('Fire Rate (%)', fontsize=12)
ax.set_title('Actual vs Predicted Fire Rate per Block', fontsize=13, fontweight='bold')
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(['B1\n2003-08', 'B2\n2009-14', 'B3\n2015-20', 'B4\n2021-25'])
ax.legend(fontsize=10)
ax.grid(alpha=0.2)

plt.suptitle('AGFE-Fire v4  Block-wise Fire Pattern Analysis', fontsize=16,
             fontweight='bold', color='#ffd700', y=1.01)
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/plots/blockwise_patterns.png', dpi=200, bbox_inches='tight',
            facecolor='black')
plt.show()
print(f" Saved: {PROJECT_DIR}/plots/blockwise_patterns.png")

Section 20 Save All Models Artifacts Locally

In [ ]:



import os, joblib

saved_files = []


model_files = [
    ('scaler.joblib', scaler),
    ('rf_model.joblib', rf_model),
    ('xgboost_model.joblib', xgb_model),
    ('catboost_model.joblib', cat_model),
    ('lightgbm_model.joblib', lgb_model),
]
for fname, obj in model_files:
    path = f'{PROJECT_DIR}/models/{fname}'
    joblib.dump(obj, path)
    size = os.path.getsize(path)
    saved_files.append((path, size))


data_files = {
    'WG_AGFE_all_blocks_engineered.csv': df_all,
}
if 'shap_values' in dir():
    data_files['shap_values.npy'] = shap_values

for fname, data in data_files.items():
    path = f'{PROJECT_DIR}/data/{fname}'
    if fname.endswith('.csv'):
        data.to_csv(path, index=False)
    else:
        np.save(path, data)
    size = os.path.getsize(path)
    saved_files.append((path, size))


feat_path = f'{PROJECT_DIR}/data/feature_list.txt'
with open(feat_path, 'w') as f:
    for feat in ALL_FEATURES:
        group = FEATURE_GROUPS.get(feat, 'Other') if 'FEATURE_GROUPS' in dir() else 'N/A'
        f.write(f"{feat}\t{group}\n")
saved_files.append((feat_path, os.path.getsize(feat_path)))


print("" * 70)
print("  ALL ARTIFACTS SAVED LOCALLY")
print("" * 70)
total_size = 0
for path, size in saved_files:
    size_str = f"{size/1024:.1f} KB" if size < 1024*1024 else f"{size/1024/1024:.1f} MB"
    print(f"   {path.replace(PROJECT_DIR+'/', ''):<45s} {size_str:>10s}")
    total_size += size
print(f"\n   Total: {total_size/1024/1024:.1f} MB across {len(saved_files)} files")
print(f"   Location: {PROJECT_DIR}/")
print("" * 70)

Section

In [ ]:



import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    f1_score, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, recall_score, precision_score
)

MODELS = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'CatBoost': cat_model,
    'LightGBM': lgb_model
}

MODEL_COLORS = {
    'Random Forest': '#4caf50',
    'XGBoost': '#448aff',
    'CatBoost': '#e91e63',
    'LightGBM': '#bb86fc',
    'S5 Ensemble': '#ffd700'
}


model_preds = {}
model_metrics = {}


thresholds = {
    'CatBoost': cat_best_thresh,
}

for name, model in MODELS.items():
    proba = model.predict_proba(X_test_scaled)[:, 1]
    thresh = thresholds.get(name, 0.5)
    pred = (proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    model_preds[name] = {'pred': pred, 'proba': proba}
    model_metrics[name] = {
        'Accuracy': accuracy_score(y_test, pred),
        'AUC-ROC': roc_auc_score(y_test, proba),
        'AUC-PR': average_precision_score(y_test, proba),
        'F1-Score': f1_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall (Sensitivity)': sens,
        'Specificity': spec,
        'MCC': matthews_corrcoef(y_test, pred),
        "Cohen's ": cohen_kappa_score(y_test, pred),
        'TSS': sens + spec - 1,
        'Brier Score': brier_score_loss(y_test, proba),
    }


tn_e, fp_e, fn_e, tp_e = confusion_matrix(y_test, ens_pred).ravel()
sens_e = tp_e / (tp_e + fn_e)
spec_e = tn_e / (tn_e + fp_e)
model_preds['S5 Ensemble'] = {'pred': ens_pred, 'proba': ens_proba}
model_metrics['S5 Ensemble'] = {
    'Accuracy': accuracy_score(y_test, ens_pred),
    'AUC-ROC': roc_auc_score(y_test, ens_proba),
    'AUC-PR': average_precision_score(y_test, ens_proba),
    'F1-Score': f1_score(y_test, ens_pred),
    'Precision': precision_score(y_test, ens_pred, zero_division=0),
    'Recall (Sensitivity)': sens_e,
    'Specificity': spec_e,
    'MCC': matthews_corrcoef(y_test, ens_pred),
    "Cohen's ": cohen_kappa_score(y_test, ens_pred),
    'TSS': sens_e + spec_e - 1,
    'Brier Score': brier_score_loss(y_test, ens_proba),
}


MODELS['S5 Ensemble'] = None

metrics_df = pd.DataFrame(model_metrics).T.round(4)
metrics_df.to_csv(f'{PROJECT_DIR}/results/model_comparison_metrics.csv')

print('' * 85)
print('   FULL MODEL COMPARISON  Block 4 Test Set (20212025)')
print('' * 85)
print(metrics_df.to_string())
print('\n   Best per metric:')
for col in metrics_df.columns:
    best = metrics_df[col].idxmin() if col == 'Brier Score' else metrics_df[col].idxmax()
    print(f'    {col:22s}: {best} ({metrics_df.loc[best, col]:.4f})')
print(f'\n Saved: {PROJECT_DIR}/results/model_comparison_metrics.csv')


In [ ]:



fig, ax = plt.subplots(figsize=(14, 6))
names = list(model_metrics.keys())
x = np.arange(len(names))
w = 0.35
acc_vals = [model_metrics[n]['Accuracy'] * 100 for n in names]
f1_vals  = [model_metrics[n]['F1-Score'] * 100 for n in names]

bars1 = ax.bar(x - w/2, acc_vals, w, label='Accuracy (%)', color='#448aff', edgecolor='white', linewidth=0.8, zorder=3)
bars2 = ax.bar(x + w/2, f1_vals, w, label='F1-Score (%)', color='#e91e63', edgecolor='white', linewidth=0.8, zorder=3)

for bars in [bars1, bars2]:
    for b in bars:
        ax.annotate(f'{b.get_height():.1f}%', xy=(b.get_x() + b.get_width()/2, b.get_height()),
                     xytext=(0, 5), textcoords='offset points', ha='center', va='bottom',
                     fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11, rotation=15, ha='right')
ax.set_ylabel('Score (%)', fontsize=13)
ax.set_title('Model Accuracy & F1-Score Comparison  AGFE-Fire v4 (with Ensemble)', fontsize=15, fontweight='bold')
ax.set_ylim(0, 110)
ax.legend(fontsize=12, loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/accuracy_f1_comparison.png', dpi=200, bbox_inches='tight')
print(' Saved: plots/accuracy_f1_comparison.png')


In [ ]:



fig, ax = plt.subplots(figsize=(9, 8))
for name in model_preds:
    fpr, tpr, _ = roc_curve(y_test, model_preds[name]['proba'])
    auc_val = model_metrics[name]['AUC-ROC']
    lw = 3.0 if name == 'Stacking Ensemble' else 2.0
    ls = '-' if name != 'Stacking Ensemble' else '--'
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.4f})",
            color=MODEL_COLORS.get(name, '#999'), linewidth=lw, linestyle=ls)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves  All Models + Ensemble', fontsize=15, fontweight='bold')
ax.legend(fontsize=10, loc='lower right', framealpha=0.9)
ax.grid(alpha=0.25)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/roc_curves_all_models.png', dpi=200, bbox_inches='tight')
print(' Saved: plots/roc_curves_all_models.png')


In [ ]:



all_names = list(model_preds.keys())
n_m = len(all_names)
fig, axes = plt.subplots(1, n_m, figsize=(5*n_m, 5))
for i, name in enumerate(all_names):
    cm = confusion_matrix(y_test, model_preds[name]['pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['No Fire', 'Fire'], yticklabels=['No Fire', 'Fire'],
                annot_kws={'size': 13}, linewidths=0.5)
    axes[i].set_title(name, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('True' if i == 0 else '', fontsize=11)
    axes[i].set_xlabel('Predicted', fontsize=11)

fig.suptitle('Confusion Matrices  All Models + Ensemble on Block 4 Test Set', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/confusion_matrices_grid.png', dpi=200, bbox_inches='tight')
print(' Saved: plots/confusion_matrices_grid.png')


In [ ]:



fig, ax = plt.subplots(figsize=(9, 8))
for name in MODELS:
    prec_vals, rec_vals, _ = precision_recall_curve(y_test, model_preds[name]['proba'])
    ap = model_metrics[name]['AUC-PR']
    ax.plot(rec_vals, prec_vals, label=f"{name} (AP = {ap:.4f})",
            color=MODEL_COLORS[name], linewidth=2.2)

fire_prevalence = y_test.mean()
ax.axhline(fire_prevalence, color='gray', linestyle='--', alpha=0.5,
           label=f'Baseline (prevalence = {fire_prevalence:.3f})')
ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('PrecisionRecall Curves  Base Models + Ensemble', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)
ax.grid(alpha=0.25)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/precision_recall_curves.png', dpi=200, bbox_inches='tight')
print(" Saved: plots/precision_recall_curves.png")

In [ ]:



radar_metrics = ['Accuracy', 'AUC-ROC', 'F1-Score', 'Precision', 'Recall (Sensitivity)', 'Specificity', 'MCC', "Cohen's "]
N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
for name in model_metrics:
    vals = [model_metrics[name][m] for m in radar_metrics]
    vals += vals[:1]
    lw = 3.0 if name == 'Stacking Ensemble' else 2.0
    ax.plot(angles, vals, 'o-', linewidth=lw, label=name, color=MODEL_COLORS.get(name, '#999'), markersize=5)
    ax.fill(angles, vals, alpha=0.06, color=MODEL_COLORS.get(name, '#999'))

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_title('Multi-Metric Radar  AGFE-Fire v4 + Ensemble', fontsize=15, fontweight='bold', y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=10, framealpha=0.9)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/radar_chart_metrics.png', dpi=200, bbox_inches='tight')
print(' Saved: plots/radar_chart_metrics.png')


In [ ]:



fig, ax = plt.subplots(figsize=(16, 6))
hm_df = metrics_df.drop(columns=['Brier Score'])
sns.heatmap(hm_df.astype(float), annot=True, fmt='.4f', cmap='YlGnBu',
            linewidths=0.6, ax=ax, vmin=0, vmax=1, annot_kws={'size': 10})
ax.set_title('Performance Heatmap  All Models + Ensemble  10 Metrics', fontsize=15, fontweight='bold')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=10)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/metric_heatmap.png', dpi=200, bbox_inches='tight')
print(' Saved: plots/metric_heatmap.png')


In [ ]:



tree_models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
}

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for i, (name, model) in enumerate(tree_models.items()):
    imps = model.feature_importances_
    idx = np.argsort(imps)[-15:]
    axes[i].barh(np.array(ALL_FEATURES)[idx], imps[idx],
                 color=MODEL_COLORS[name], edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'{name}  Top 15 Features', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Importance', fontsize=11)
    axes[i].tick_params(axis='y', labelsize=10)
    axes[i].invert_yaxis()

fig.suptitle('Feature Importance Comparison  Tree-Based Models', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/feature_importance_comparison.png', dpi=200, bbox_inches='tight')
print(" Saved: plots/feature_importance_comparison.png")

Section 22 Ensemble Comparison Plots

Section 22a Integrated Mapping Pipeline Moved from Standalone Scripts

In [ ]:



import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import accuracy_score, f1_score

os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/plots', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/maps', exist_ok=True)


test_mask = df_all['block_id'] == 4
test_df_coords = df_all.loc[test_mask, ['latitude', 'longitude', 'block_id', TARGET_COL]].copy().reset_index(drop=True)

n = min(len(test_df_coords), len(y_test), len(X_test_scaled))
test_df_coords = test_df_coords.head(n).copy()
y_true_map = y_test[:n]

results_coords = pd.DataFrame({
    'latitude': test_df_coords['latitude'].values,
    'longitude': test_df_coords['longitude'].values,
    'y_true': y_true_map,
})


results_coords['pred_RF'] = rf_model.predict(X_test_scaled[:n])
results_coords['prob_RF'] = rf_model.predict_proba(X_test_scaled[:n])[:, 1]
results_coords['pred_XGBoost'] = xgb_model.predict(X_test_scaled[:n])
results_coords['prob_XGBoost'] = xgb_model.predict_proba(X_test_scaled[:n])[:, 1]
results_coords['pred_LightGBM'] = lgb_model.predict(X_test_scaled[:n])
results_coords['prob_LightGBM'] = lgb_model.predict_proba(X_test_scaled[:n])[:, 1]

if 'cat_model' in globals() and cat_model is not None:
    cat_thr_map = cat_best_thresh if 'cat_best_thresh' in globals() else 0.5
    cat_proba_map = cat_model.predict_proba(X_test_scaled[:n])[:, 1]
    results_coords['prob_CatBoost'] = cat_proba_map
    results_coords['pred_CatBoost'] = (cat_proba_map >= cat_thr_map).astype(int)


results_coords['prob_Top3Ensemble'] = ens_proba[:n]
results_coords['pred_Top3Ensemble'] = ens_pred[:n]


results_coords['confusion_cat'] = 'TN'
results_coords.loc[(results_coords['y_true'] == 1) & (results_coords['pred_Top3Ensemble'] == 1), 'confusion_cat'] = 'TP'
results_coords.loc[(results_coords['y_true'] == 1) & (results_coords['pred_Top3Ensemble'] == 0), 'confusion_cat'] = 'FN'
results_coords.loc[(results_coords['y_true'] == 0) & (results_coords['pred_Top3Ensemble'] == 1), 'confusion_cat'] = 'FP'


agreement_cols = [c for c in ['pred_RF', 'pred_XGBoost', 'pred_LightGBM', 'pred_CatBoost'] if c in results_coords.columns]
results_coords['model_agreement'] = results_coords[agreement_cols].sum(axis=1)


results_coords.to_csv(f'{PROJECT_DIR}/results/test_results_with_coords.csv', index=False)
all_blocks_coords = df_all[['latitude', 'longitude', 'block_id', TARGET_COL]].copy()
all_blocks_coords.to_csv(f'{PROJECT_DIR}/results/all_blocks_coords.csv', index=False)
print(' Saved: results/test_results_with_coords.csv')
print(' Saved: results/all_blocks_coords.csv')


WG_XLIM = (73.0, 78.5)
WG_YLIM = (8.0, 21.5)

def make_map_ax(title, figsize=(10, 14)):
    fig, ax = plt.subplots(1, 1, figsize=figsize, facecolor='#0F172A')
    ax.set_facecolor('#1E293B')
    ax.set_title(title, fontsize=16, fontweight='bold', color='white', pad=12)
    ax.set_xlabel('Longitude', fontsize=11, color='#94A3B8')
    ax.set_ylabel('Latitude', fontsize=11, color='#94A3B8')
    ax.tick_params(colors='#64748B', labelsize=9)
    ax.set_xlim(*WG_XLIM)
    ax.set_ylim(*WG_YLIM)
    for spine in ax.spines.values():
        spine.set_color('#334155')
    return fig, ax

df_map = results_coords.copy()


fig, ax = make_map_ax('Ground Truth  Block 4 (2021-2025) Fire Occurrence')
no_fire = df_map[df_map['y_true'] == 0]
fire = df_map[df_map['y_true'] == 1]
ax.scatter(no_fire['longitude'], no_fire['latitude'], c='#64748B', s=4, alpha=0.35, label=f'No Fire ({len(no_fire):,})')
ax.scatter(fire['longitude'], fire['latitude'], c='#3B82F6', s=8, alpha=0.85, label=f'Fire ({len(fire):,})')
ax.legend(loc='lower left', fontsize=10, facecolor='#1E293B', edgecolor='#475569', labelcolor='white')
fig.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/map_ground_truth_block4.png', dpi=300, bbox_inches='tight')
plt.close(fig)


fig, ax = make_map_ax('Top-3 Ensemble Prediction  Block 4 (2021-2025)')
pred_no = df_map[df_map['pred_Top3Ensemble'] == 0]
pred_yes = df_map[df_map['pred_Top3Ensemble'] == 1]
ax.scatter(pred_no['longitude'], pred_no['latitude'], c='#64748B', s=4, alpha=0.35, label=f'Pred No Fire ({len(pred_no):,})')
ax.scatter(pred_yes['longitude'], pred_yes['latitude'], c='#EF4444', s=8, alpha=0.85, label=f'Pred Fire ({len(pred_yes):,})')
ax.legend(loc='lower left', fontsize=10, facecolor='#1E293B', edgecolor='#475569', labelcolor='white')
fig.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/map_predicted_block4.png', dpi=300, bbox_inches='tight')
plt.close(fig)


fig, ax = make_map_ax('Spatial Error Analysis  TP / TN / FP / FN')
cat_cfg = {
    'TN': ('#475569', 3, 0.25),
    'TP': ('#22C55E', 8, 0.75),
    'FN': ('#F97316', 11, 0.9),
    'FP': ('#EF4444', 11, 0.9),
}
for cat in ['TN', 'TP', 'FN', 'FP']:
    d = df_map[df_map['confusion_cat'] == cat]
    c, s, a = cat_cfg[cat]
    ax.scatter(d['longitude'], d['latitude'], c=c, s=s, alpha=a, label=f'{cat} ({len(d):,})')
ax.legend(loc='lower left', fontsize=10, facecolor='#1E293B', edgecolor='#475569', labelcolor='white')
fig.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/map_confusion_spatial.png', dpi=300, bbox_inches='tight')
plt.close(fig)


fig, ax = make_map_ax('Fire Probability  Top-3 Ensemble (Block 4)')
sc = ax.scatter(df_map['longitude'], df_map['latitude'], c=df_map['prob_Top3Ensemble'], cmap='YlOrRd', s=7, alpha=0.85, vmin=0, vmax=1)
cbar = fig.colorbar(sc, ax=ax, shrink=0.65, pad=0.02, aspect=30)
cbar.set_label('Fire Probability', color='white', fontsize=11)
cbar.ax.tick_params(colors='white', labelsize=9)
fig.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/map_fire_probability.png', dpi=300, bbox_inches='tight')
plt.close(fig)


fig, ax = make_map_ax(f"Model Agreement  Individual Models Predicting Fire (0-{len(agreement_cols)})")
agree_cmap = LinearSegmentedColormap.from_list('agree', ['#F1F5F9', '#FDE68A', '#FBBF24', '#F97316', '#DC2626', '#7F1D1D'], N=max(6, len(agreement_cols)+1))
sc = ax.scatter(df_map['longitude'], df_map['latitude'], c=df_map['model_agreement'], cmap=agree_cmap, s=7, alpha=0.85, vmin=0, vmax=max(1, len(agreement_cols)))
cbar = fig.colorbar(sc, ax=ax, shrink=0.65, pad=0.02, aspect=30)
cbar.set_label(f'Models Agreeing (out of {len(agreement_cols)})', color='white', fontsize=11)
cbar.ax.tick_params(colors='white', labelsize=9)
fig.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/map_model_agreement.png', dpi=300, bbox_inches='tight')
plt.close(fig)


model_grid = {
    'Random Forest': 'pred_RF',
    'XGBoost': 'pred_XGBoost',
    'LightGBM': 'pred_LightGBM',
    'Top-3 Ensemble': 'pred_Top3Ensemble',
}
fig, axes = plt.subplots(2, 2, figsize=(16, 22), facecolor='#0F172A')
for ax, (name, col) in zip(axes.flatten(), model_grid.items()):
    ax.set_facecolor('#1E293B')
    d0 = df_map[df_map[col] == 0]
    d1 = df_map[df_map[col] == 1]
    ax.scatter(d0['longitude'], d0['latitude'], c='#475569', s=3, alpha=0.3)
    ax.scatter(d1['longitude'], d1['latitude'], c='#FF4D4D', s=6, alpha=0.75)
    acc_m = accuracy_score(df_map['y_true'], df_map[col])
    f1_m = f1_score(df_map['y_true'], df_map[col])
    ax.set_title(f'{name}\nAcc={acc_m*100:.1f}%  F1={f1_m:.3f}', fontsize=11, fontweight='bold', color='white')
    ax.set_xlim(*WG_XLIM)
    ax.set_ylim(*WG_YLIM)
    ax.tick_params(colors='#64748B', labelsize=8)
    for spine in ax.spines.values():
        spine.set_color('#334155')
fig.suptitle('Per-Model Fire Prediction Comparison  Block 4 (2021-2025)', fontsize=18, fontweight='bold', color='white', y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.98])
fig.savefig(f'{PROJECT_DIR}/plots/map_model_comparison.png', dpi=300, bbox_inches='tight')
plt.close(fig)


hist = pd.read_csv(f'{PROJECT_DIR}/results/all_blocks_coords.csv')
block_labels = {1: 'Block 1 (2003-2008)', 2: 'Block 2 (2009-2014)', 3: 'Block 3 (2015-2020)', 4: 'Block 4 (2021-2025)'}
fig, axes = plt.subplots(2, 2, figsize=(16, 22), facecolor='#0F172A')
for ax, bid in zip(axes.flatten(), [1, 2, 3, 4]):
    ax.set_facecolor('#1E293B')
    d = hist[hist['block_id'] == bid]
    d0 = d[d[TARGET_COL] == 0]
    d1 = d[d[TARGET_COL] == 1]
    ax.scatter(d0['longitude'], d0['latitude'], c='#475569', s=2.5, alpha=0.30, label=f'No Fire ({len(d0):,})')
    ax.scatter(d1['longitude'], d1['latitude'], c='#FF4D4D', s=4.5, alpha=0.75, label=f'Fire ({len(d1):,})')
    ax.set_title(f"{block_labels[bid]}\n{len(d):,} points | Fire rate: {d[TARGET_COL].mean():.1%}", fontsize=10, fontweight='bold', color='white')
    ax.set_xlim(*WG_XLIM)
    ax.set_ylim(*WG_YLIM)
    ax.tick_params(colors='#64748B', labelsize=8)
    for spine in ax.spines.values():
        spine.set_color('#334155')
    ax.legend(loc='lower left', fontsize=6, facecolor='#1E293B', edgecolor='#475569', labelcolor='white')
fig.suptitle('22-Year Fire History  Western Ghats (2003-2025)', fontsize=18, fontweight='bold', color='white', y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.98])
fig.savefig(f'{PROJECT_DIR}/plots/map_fire_history_4blocks.png', dpi=300, bbox_inches='tight')
plt.close(fig)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 14), facecolor='#0F172A')
for ax in [ax1, ax2]:
    ax.set_facecolor('#1E293B')
    ax.set_xlim(*WG_XLIM)
    ax.set_ylim(*WG_YLIM)
    ax.tick_params(colors='#64748B', labelsize=9)
    for spine in ax.spines.values():
        spine.set_color('#334155')

ax1.scatter(no_fire['longitude'], no_fire['latitude'], c='#64748B', s=4, alpha=0.35)
ax1.scatter(fire['longitude'], fire['latitude'], c='#3B82F6', s=8, alpha=0.85)
ax1.set_title(f'Ground Truth\nFire: {len(fire):,} | No Fire: {len(no_fire):,}', fontsize=14, fontweight='bold', color='white')

acc_s5 = accuracy_score(df_map['y_true'], df_map['pred_Top3Ensemble'])
ax2.scatter(pred_no['longitude'], pred_no['latitude'], c='#64748B', s=4, alpha=0.35)
ax2.scatter(pred_yes['longitude'], pred_yes['latitude'], c='#EF4444', s=8, alpha=0.85)
ax2.set_title(f'Top-3 Ensemble Prediction (Acc: {acc_s5:.1%})\nPred Fire: {len(pred_yes):,} | Pred No Fire: {len(pred_no):,}', fontsize=14, fontweight='bold', color='white')

fig.suptitle('AGFE-Fire v4  Ground Truth vs Prediction  Block 4 (2021-2025)', fontsize=18, fontweight='bold', color='white', y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f'{PROJECT_DIR}/plots/map_gt_vs_predicted_sidebyside.png', dpi=300, bbox_inches='tight')
plt.close(fig)


try:
    import folium
    from folium.plugins import HeatMap

    center = [12.5, 76.0]


    m = folium.Map(location=center, zoom_start=7, tiles='CartoDB positron', control_scale=True, prefer_canvas=True)
    gt_layer = folium.FeatureGroup(name='Ground Truth', show=True)
    for _, r in df_map.iterrows():
        color = '#2563EB' if r['y_true'] == 1 else '#94A3B8'
        folium.CircleMarker(location=[r['latitude'], r['longitude']], radius=3 if r['y_true'] == 1 else 2,
                            color=color, fill=True, fill_color=color, fill_opacity=0.7, weight=0.5).add_to(gt_layer)
    gt_layer.add_to(m)

    pred_layer = folium.FeatureGroup(name='Top-3 Ensemble', show=False)
    for _, r in df_map.iterrows():
        color = '#DC2626' if r['pred_Top3Ensemble'] == 1 else '#D1D5DB'
        folium.CircleMarker(location=[r['latitude'], r['longitude']], radius=3 if r['pred_Top3Ensemble'] == 1 else 2,
                            color=color, fill=True, fill_color=color, fill_opacity=0.7, weight=0.5).add_to(pred_layer)
    pred_layer.add_to(m)

    conf_layer = folium.FeatureGroup(name='Confusion TP/FP/FN/TN', show=False)
    conf_color = {'TP': '#16A34A', 'TN': '#E2E8F0', 'FP': '#DC2626', 'FN': '#F97316'}
    for _, r in df_map.iterrows():
        c = conf_color[r['confusion_cat']]
        folium.CircleMarker(location=[r['latitude'], r['longitude']], radius=4 if r['confusion_cat'] in ['FP', 'FN'] else 3,
                            color=c, fill=True, fill_color=c, fill_opacity=0.8 if r['confusion_cat'] in ['FP', 'FN'] else 0.5, weight=0.5).add_to(conf_layer)
    conf_layer.add_to(m)

    heat_layer = folium.FeatureGroup(name='Fire Probability Heatmap', show=False)
    heat_data = df_map[df_map['prob_Top3Ensemble'] > 0.3][['latitude', 'longitude', 'prob_Top3Ensemble']].values.tolist()
    HeatMap(heat_data, min_opacity=0.3, radius=12, blur=8,
            gradient={0.3: '#FBBF24', 0.5: '#F97316', 0.7: '#DC2626', 1.0: '#7F1D1D'}).add_to(heat_layer)
    heat_layer.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save(f'{PROJECT_DIR}/maps/ground_truth_vs_predicted.html')


    m2 = folium.Map(location=center, zoom_start=7, tiles='CartoDB positron', control_scale=True)
    for bid in [1, 2, 3, 4]:
        d = hist[hist['block_id'] == bid]
        layer = folium.FeatureGroup(name=block_labels[bid], show=(bid == 4))
        for _, r in d.iterrows():
            color = '#FF4D4D' if r[TARGET_COL] == 1 else '#64748B'
            folium.CircleMarker(location=[r['latitude'], r['longitude']], radius=2 if r[TARGET_COL] == 1 else 1,
                                color=color, fill=True, fill_color=color, fill_opacity=0.65, weight=0.2).add_to(layer)
        layer.add_to(m2)
    folium.LayerControl(collapsed=False).add_to(m2)
    m2.save(f'{PROJECT_DIR}/maps/fire_history_all_blocks.html')

    print(' Saved: maps/ground_truth_vs_predicted.html')
    print(' Saved: maps/fire_history_all_blocks.html')
except Exception as e:
    print(f' Folium interactive map generation skipped: {e}')

print(' Section 22a complete: standalone-only map workflow is now in notebook.')

In [ ]:










import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    f1_score, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss, confusion_matrix
)
from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import rankdata

print(' Section 22: Ensemble Comparison Plots')
print('' * 60)


def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'AUC-ROC': roc_auc_score(y_true, y_prob),
        'AUC-PR': average_precision_score(y_true, y_prob),
        'F1-Score': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        "Cohen's ": cohen_kappa_score(y_true, y_pred),
        'TSS': sensitivity + specificity - 1,
        'Brier Score': brier_score_loss(y_true, y_prob),
        'Sensitivity': sensitivity,
        'Specificity': specificity,
    }


probas_dict = {
    'RF': rf_model.predict_proba(X_test_scaled)[:, 1],
    'XGBoost': xgb_model.predict_proba(X_test_scaled)[:, 1],
    'LightGBM': lgb_model.predict_proba(X_test_scaled)[:, 1],
}


all_model_names = ['RF', 'XGBoost', 'LightGBM']
all_models_dict = {'RF': rf_model, 'XGBoost': xgb_model, 'LightGBM': lgb_model}
try:
    cat_model_loaded = joblib.load(f'{PROJECT_DIR}/models/catboost_model.joblib')
    probas_dict['CatBoost'] = cat_model_loaded.predict_proba(X_test_scaled)[:, 1]
    all_models_dict['CatBoost'] = cat_model_loaded
    all_model_names.append('CatBoost')
except: pass

preds_dict = {n: m.predict(X_test_scaled) for n, m in all_models_dict.items()}


top3_proba = (probas_dict['RF'] + probas_dict['XGBoost'] + probas_dict['LightGBM']) / 3.0
top3_pred = (top3_proba >= 0.5).astype(int)
metrics_A = compute_metrics(y_test, top3_pred, top3_proba)
print(f'  A: Top-3 Average     Acc={metrics_A["Accuracy"]:.4f} F1={metrics_A["F1-Score"]:.4f}')


auc_w = {n: roc_auc_score(y_test, probas_dict[n]) for n in ['RF','XGBoost','LightGBM']}
total_w = sum(auc_w.values())
weighted_proba = sum(probas_dict[m] * (w / total_w) for m, w in auc_w.items())
weighted_pred = (weighted_proba >= 0.5).astype(int)
metrics_B = compute_metrics(y_test, weighted_pred, weighted_proba)
print(f'  B: Top-3 Weighted    Acc={metrics_B["Accuracy"]:.4f} F1={metrics_B["F1-Score"]:.4f}')


top3_names = ['RF', 'XGBoost', 'LightGBM']
oof_train_proba = np.column_stack([all_models_dict[m].predict_proba(X_train_scaled)[:, 1] for m in top3_names])
oof_test_proba = np.column_stack([probas_dict[m] for m in top3_names])
meta_gb = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
meta_gb.fit(oof_train_proba, y_train_sm)
stack_proba = meta_gb.predict_proba(oof_test_proba)[:, 1]
stack_pred = meta_gb.predict(oof_test_proba)
metrics_C = compute_metrics(y_test, stack_pred, stack_proba)
print(f'  C: Top-3 Stacking    Acc={metrics_C["Accuracy"]:.4f} F1={metrics_C["F1-Score"]:.4f}')


soft_proba = np.mean([probas_dict[m] for m in all_model_names], axis=0)
soft_pred = (soft_proba >= 0.5).astype(int)
metrics_D = compute_metrics(y_test, soft_pred, soft_proba)
print(f'  D: All Soft Vote     Acc={metrics_D["Accuracy"]:.4f} F1={metrics_D["F1-Score"]:.4f}')


rank_proba = np.mean([rankdata(probas_dict[m]) / len(y_test) for m in top3_names], axis=0)
rank_pred = (rank_proba >= 0.5).astype(int)
metrics_E = compute_metrics(y_test, rank_pred, rank_proba)
print(f'  E: Rank Fusion       Acc={metrics_E["Accuracy"]:.4f} F1={metrics_E["F1-Score"]:.4f}')


try:
    meta_lr_loaded = joblib.load(f'{PROJECT_DIR}/models/meta_learner.joblib')
    oof_test_loaded = np.load(f'{PROJECT_DIR}/data/oof_test.npy')
    orig_proba = meta_lr_loaded.predict_proba(oof_test_loaded)[:, 1]
    orig_pred = meta_lr_loaded.predict(oof_test_loaded)
    metrics_orig = compute_metrics(y_test, orig_pred, orig_proba)
    print(f'  Original Ensemble    Acc={metrics_orig["Accuracy"]:.4f} F1={metrics_orig["F1-Score"]:.4f}')
except:
    metrics_orig = metrics_A
    print('   Original meta-learner not found, using Strategy A as fallback')


individual_metrics = {n: compute_metrics(y_test, preds_dict[n], probas_dict[n]) for n in all_model_names}


all_results = {}
for name in all_model_names:
    all_results[name] = individual_metrics[name]
all_results['Original Ensemble'] = metrics_orig
all_results['A: Top-3 Average'] = metrics_A
all_results['B: Top-3 Weighted'] = metrics_B
all_results['C: Top-3 Stacking'] = metrics_C
all_results['D: All Soft Vote'] = metrics_D
all_results['E: Rank Fusion'] = metrics_E

comparison = pd.DataFrame(all_results).T
comparison = comparison.sort_values('Accuracy', ascending=False)
print(f'\n{comparison.to_string(float_format="%.4f")}')
comparison.to_csv(f'{PROJECT_DIR}/results/ensemble_comparison.csv')
print(f'\n Saved: results/ensemble_comparison.csv')




print('\n Creating ensemble comparison bar chart...')
ens_names = ['Original Ensemble', 'A: Top-3 Average', 'B: Top-3 Weighted',
             'C: Top-3 Stacking', 'D: All Soft Vote', 'E: Rank Fusion']
ind_names = [n for n in comparison.index if n not in ens_names]

fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor='#0F172A')


ax1 = axes[0]
ax1.set_facecolor('#1E293B')
ind_data = comparison.loc[[n for n in ind_names if n in comparison.index]]
colors_ind = ['#3B82F6', '#22C55E', '#F97316', '#8B5CF6', '#EF4444']
bars = ax1.barh(ind_data.index, ind_data['Accuracy'] * 100,
                color=colors_ind[:len(ind_data)], height=0.5, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, ind_data['Accuracy'] * 100):
    ax1.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.2f}%',
             va='center', fontsize=10, color='white', fontweight='bold')
ax1.set_xlim(70, 100)
ax1.set_title('Individual Models', fontsize=14, fontweight='bold', color='white', pad=12)
ax1.set_xlabel('Accuracy (%)', color='#94A3B8', fontsize=11)
ax1.tick_params(colors='#CBD5E1', labelsize=10)
ax1.axvline(x=metrics_A['Accuracy'] * 100, color='#FBBF24', linestyle='--', alpha=0.7, label='Best Ensemble')
ax1.legend(facecolor='#1E293B', edgecolor='#475569', labelcolor='white', fontsize=9)
for spine in ax1.spines.values(): spine.set_color('#334155')


ax2 = axes[1]
ax2.set_facecolor('#1E293B')
ens_data = comparison.loc[[n for n in ens_names if n in comparison.index]]
colors_ens = ['#94A3B8', '#22C55E', '#10B981', '#FBBF24', '#8B5CF6', '#F97316']
bars = ax2.barh(ens_data.index, ens_data['Accuracy'] * 100,
                color=colors_ens[:len(ens_data)], height=0.5, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, ens_data['Accuracy'] * 100):
    ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.2f}%',
             va='center', fontsize=10, color='white', fontweight='bold')
ax2.set_xlim(70, 100)
ax2.set_title('Ensemble Strategies', fontsize=14, fontweight='bold', color='white', pad=12)
ax2.set_xlabel('Accuracy (%)', color='#94A3B8', fontsize=11)
ax2.tick_params(colors='#CBD5E1', labelsize=10)
ax2.axvline(x=individual_metrics['RF']['Accuracy'] * 100, color='#3B82F6', linestyle='--', alpha=0.7, label='Best Individual (RF)')
ax2.legend(facecolor='#1E293B', edgecolor='#475569', labelcolor='white', fontsize=9)
for spine in ax2.spines.values(): spine.set_color('#334155')

fig.suptitle('Model & Ensemble Performance Comparison  Block 4 Test Set',
             fontsize=16, fontweight='bold', color='white', y=1.02)
fig.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/ensemble_comparison_bars.png', dpi=300, bbox_inches='tight')
plt.close()
print('   plots/ensemble_comparison_bars.png')




print(' Creating metrics heatmap...')
key_metrics = ['Accuracy', 'AUC-ROC', 'F1-Score', 'MCC', 'Sensitivity', 'Specificity']
heatmap_data = comparison[key_metrics]

fig, ax = plt.subplots(figsize=(12, 8), facecolor='#0F172A')
ax.set_facecolor('#1E293B')
cmap = LinearSegmentedColormap.from_list('custom', ['#7F1D1D', '#DC2626', '#F97316', '#FBBF24', '#22C55E', '#059669'], N=256)
im = ax.imshow(heatmap_data.values, cmap=cmap, aspect='auto', vmin=0.4, vmax=1.0)
for i in range(len(heatmap_data)):
    for j in range(len(key_metrics)):
        val = heatmap_data.iloc[i, j]
        text_color = 'white' if val < 0.7 else '#0F172A'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=9, fontweight='bold', color=text_color)
ax.set_xticks(range(len(key_metrics)))
ax.set_xticklabels(key_metrics, fontsize=10, color='#CBD5E1', rotation=30, ha='right')
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels(heatmap_data.index, fontsize=10, color='#CBD5E1')
ax.set_title('Complete Metrics Comparison  All Strategies', fontsize=14, fontweight='bold', color='white', pad=15)
cbar = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('Score', color='white', fontsize=11)
cbar.ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_color('#334155')
fig.tight_layout()
fig.savefig(f'{PROJECT_DIR}/plots/ensemble_metrics_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print('   plots/ensemble_metrics_heatmap.png')




print('  Creating GT vs RF vs Best Ensemble map...')
best_ens_strategies = {
    'A: Top-3 Average': (top3_pred, top3_proba),
    'B: Top-3 Weighted': (weighted_pred, weighted_proba),
    'C: Top-3 Stacking': (stack_pred, stack_proba),
}
best_ens_name = max(['A: Top-3 Average', 'B: Top-3 Weighted', 'C: Top-3 Stacking'],
                    key=lambda n: all_results[n]['Accuracy'])
best_ens_pred, best_ens_proba = best_ens_strategies[best_ens_name]

test_coords_df = pd.read_csv(f'{PROJECT_DIR}/results/test_results_with_coords.csv')

fig, axes = plt.subplots(1, 3, figsize=(24, 14), facecolor='#0F172A')
for ax in axes:
    ax.set_facecolor('#1E293B')
    ax.set_xlim(73.0, 78.5)
    ax.set_ylim(8.0, 21.5)
    ax.tick_params(colors='#64748B', labelsize=8)
    for spine in ax.spines.values(): spine.set_color('#334155')


ax = axes[0]
no_f = test_coords_df[test_coords_df['y_true'] == 0]
yes_f = test_coords_df[test_coords_df['y_true'] == 1]
ax.scatter(no_f['longitude'], no_f['latitude'], c='#475569', s=4, alpha=0.3)
ax.scatter(yes_f['longitude'], yes_f['latitude'], c='#3B82F6', s=8, alpha=0.8)
ax.set_title(f'Ground Truth\nFire: {len(yes_f):,} | No-fire: {len(no_f):,}',
             fontsize=13, fontweight='bold', color='white', pad=10)


ax = axes[1]
rf_no = test_coords_df[test_coords_df['pred_RF'] == 0]
rf_yes = test_coords_df[test_coords_df['pred_RF'] == 1]
rf_acc = accuracy_score(y_test, preds_dict['RF'])
ax.scatter(rf_no['longitude'], rf_no['latitude'], c='#475569', s=4, alpha=0.3)
ax.scatter(rf_yes['longitude'], rf_yes['latitude'], c='#EF4444', s=8, alpha=0.8)
ax.set_title(f'Best Individual: RF (Acc={rf_acc:.1%})\nPred Fire: {len(rf_yes):,}',
             fontsize=13, fontweight='bold', color='white', pad=10)


ax = axes[2]
best_acc = accuracy_score(y_test, best_ens_pred)
ens_no = test_coords_df.loc[best_ens_pred == 0]
ens_yes = test_coords_df.loc[best_ens_pred == 1]
ax.scatter(ens_no['longitude'], ens_no['latitude'], c='#475569', s=4, alpha=0.3)
ax.scatter(ens_yes['longitude'], ens_yes['latitude'], c='#FBBF24', s=8, alpha=0.8)
ax.set_title(f'Best Ensemble: {best_ens_name} (Acc={best_acc:.1%})\nPred Fire: {int(best_ens_pred.sum()):,}',
             fontsize=13, fontweight='bold', color='white', pad=10)

fig.suptitle('GT vs Best Individual vs Best Ensemble  Block 4 (20212025)',
             fontsize=18, fontweight='bold', color='white', y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f'{PROJECT_DIR}/plots/map_ensemble_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print('   plots/map_ensemble_comparison.png')


if best_ens_name == 'C: Top-3 Stacking':
    joblib.dump(meta_gb, f'{PROJECT_DIR}/models/best_ensemble.joblib')
    print(f'\n Saved: models/best_ensemble.joblib')

print(f'\n Best Ensemble: {best_ens_name} (Acc={all_results[best_ens_name]["Accuracy"]:.4f})')
print(' Section 22 complete!')


Section 23 Future Fire Predictions Block 5 20262031

In [ ]:










import json as jsonmod
from scipy.spatial import cKDTree

print(' Section 23: Future Fire Predictions')
print('' * 60)


BLOCK5_CSV = f'{PROJECT_DIR}/WG_AGFE_Block5_2026_2031(4).csv'
BLOCK4_CSV = f'{PROJECT_DIR}/WG_AGFE_Block4_2021_2025.csv'


W_RF, W_XGB, W_LGB = 0.45, 0.05, 0.50
THRESHOLD = 0.315


with open(f'{PROJECT_DIR}/data/ALL_FEATURES.json') as f:
    ALL_FEATURES = jsonmod.load(f)
print(f'  Features: {len(ALL_FEATURES)}')
print(f'  Ensemble: RF({W_RF}) + XGB({W_XGB}) + LGB({W_LGB}), threshold={THRESHOLD}')


print(f'\n Loading Block 5: {BLOCK5_CSV}')
df5 = pd.read_csv(BLOCK5_CSV)
if len(df5) < 10:
    raise ValueError(f'Block 5 CSV has only {len(df5)} rows! Run the GEE script first.')
print(f'  Rows: {len(df5)}, Columns: {len(df5.columns)}')
coords = df5[['longitude','latitude']].copy()


print(f' Loading Block 4: {BLOCK4_CSV}')
df4 = pd.read_csv(BLOCK4_CSV)
print(f'  Block 4: {len(df4)} rows')


print('\n Feature Engineering...')


env_features = ['FWI_proxy', 'NDVI_min', 'aspect_deg', 'elevation',
                'land_cover', 'population_density', 'precip_total_mm', 'terrain_x_veg']
for feat in env_features:
    if feat not in df5.columns:
        print(f'   Missing: {feat} (setting to 0)')
        df5[feat] = 0


if 'longitude' in df4.columns and 'latitude' in df4.columns:
    tree4 = cKDTree(df4[['longitude','latitude']].values)
    dists, idxs = tree4.query(df5[['longitude','latitude']].values, k=1)
    fire_cols_from_prev = {
        'fire_occurrence': 'prev_fire_occurrence',
        'total_fire_count': 'prev_total_fire_count',
        'FRP_mean_MW': 'prev_FRP_mean_MW',
        'FRP_max_MW': 'prev_FRP_max_MW',
        'burned_area_binary': 'prev_burned_area_binary',
        'burn_month_count': 'prev_burn_month_count',
        'FIRMS_brightness_K': 'prev_FIRMS_brightness_K',
    }
    for src, dst in fire_cols_from_prev.items():
        if src in df4.columns:
            df5[dst] = df4.iloc[idxs][src].values
            print(f'   {dst}  Block4.{src}')
        else:
            df5[dst] = 0
            print(f'   {dst} = 0 (not in Block 4)')
else:
    print('   No coords in Block 4')
    for feat in ALL_FEATURES:
        if feat.startswith('prev_') and feat not in df5.columns:
            df5[feat] = 0


df5['fire_recurrence'] = df5.get('prev_total_fire_count', 0)
df5['fire_trend_delta'] = 0


if 'prev_fire_occurrence' in df5.columns:
    fire_pts = df5[df5['prev_fire_occurrence'] == 1][['longitude','latitude']]
    if len(fire_pts) > 0:
        fire_tree = cKDTree(fire_pts.values)
        dists_fire, _ = fire_tree.query(df5[['longitude','latitude']].values, k=1)
        df5['neighbor_fire_5km'] = (dists_fire <= 0.045).astype(int)
    else:
        df5['neighbor_fire_5km'] = 0
else:
    df5['neighbor_fire_5km'] = 0


delta_base_feats = ['FWI_proxy','NDVI_min','aspect_deg','elevation',
                    'land_cover','population_density','precip_total_mm','terrain_x_veg']
for feat in delta_base_feats:
    delta_col = f'delta_{feat}'
    if feat in df4.columns and feat in df5.columns and 'idxs' in dir():
        prev_vals = df4.iloc[idxs][feat].values
        df5[delta_col] = df5[feat].values - prev_vals
        print(f'   {delta_col}')
    else:
        df5[delta_col] = 0


df5['ONI_index'] = 0.0
df5['IOD_index'] = 0.0
print('   ONI_index, IOD_index set to neutral (0.0)')


df5['fwi_x_ndvi'] = df5['FWI_proxy'] * df5['NDVI_min']
df5['prev_frp_x_fwi'] = df5.get('prev_FRP_mean_MW', 0) * df5['FWI_proxy']
df5['elev_x_ndvi'] = df5['elevation'] * df5['NDVI_min']
df5['oni_x_precip'] = df5['ONI_index'] * df5['precip_total_mm']
df5['neighbor_fire_x_fwi'] = df5['neighbor_fire_5km'] * df5['FWI_proxy']
print('   Interaction features')


df5['fire_season'] = 1
print('   fire_season = 1 (fire season assumed)')


print(f'\n Aligning {len(ALL_FEATURES)} features...')
missing = [f for f in ALL_FEATURES if f not in df5.columns]
if missing:
    print(f'   Missing {len(missing)} features (filling with 0): {missing}')
    for f in missing:
        df5[f] = 0

X_future = df5[ALL_FEATURES].values.astype(float)
X_future = np.nan_to_num(X_future, nan=0.0, posinf=0.0, neginf=0.0)
print(f'  Shape: {X_future.shape}')
X_future_scaled = scaler.transform(X_future)


print('\n Predicting with Ensemble...')
rf_proba_f  = rf_model.predict_proba(X_future_scaled)[:, 1]
xgb_proba_f = xgb_model.predict_proba(X_future_scaled)[:, 1]
lgb_proba_f = lgb_model.predict_proba(X_future_scaled)[:, 1]

ens_proba_f = W_RF * rf_proba_f + W_XGB * xgb_proba_f + W_LGB * lgb_proba_f
ens_pred_f = (ens_proba_f >= THRESHOLD).astype(int)

n_fire = ens_pred_f.sum()
n_total = len(ens_pred_f)
fire_pct = n_fire / n_total * 100

print(f'\n{"" * 60}')
print(f'   FUTURE FIRE PREDICTIONS (Block 5: 2026-2031)')
print(f'{"" * 60}')
print(f'  Total sample points : {n_total:,}')
print(f'  Predicted FIRE      : {n_fire:,} ({fire_pct:.1f}%)')
print(f'  Predicted NO FIRE   : {n_total-n_fire:,} ({100-fire_pct:.1f}%)')
print(f'  Ensemble weights    : RF={W_RF}, XGB={W_XGB}, LGB={W_LGB}')
print(f'  Threshold           : {THRESHOLD}')
print(f'{"" * 60}')


results_future = coords.copy()
results_future['pred_fire'] = ens_pred_f
results_future['prob_fire'] = ens_proba_f.round(4)
results_future['prob_RF'] = rf_proba_f.round(4)
results_future['prob_XGB'] = xgb_proba_f.round(4)
results_future['prob_LGB'] = lgb_proba_f.round(4)
results_future['risk_level'] = pd.cut(ens_proba_f,
                                bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
                                labels=['Very Low','Low','Medium','High','Very High'])

os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
out_csv = f'{PROJECT_DIR}/results/future_predictions_2026_2031.csv'
results_future.to_csv(out_csv, index=False)
print(f'\n Saved: {out_csv}')


print('\n Generating GEE visualization script...')
fire_coords = results_future[results_future['pred_fire']==1][['longitude','latitude']].values.tolist()
nofire_coords = results_future[results_future['pred_fire']==0][['longitude','latitude']].values.tolist()
high_risk = results_future[results_future['prob_fire'] >= 0.7][['longitude','latitude']].values.tolist()
medium_risk = results_future[(results_future['prob_fire'] >= 0.4) & (results_future['prob_fire'] < 0.7)][['longitude','latitude']].values.tolist()

gee_script = f'''// =============================================================
// AGFE-Fire v4  |  FUTURE Fire Predictions 2026-2031
// =============================================================
//    RED    = Predicted FIRE locations     ({len(fire_coords)} pts)
//    GREEN  = Predicted NO FIRE            ({len(nofire_coords)} pts)
//    ORANGE = HIGH risk (prob  0.7)       ({len(high_risk)} pts)
//    YELLOW = MEDIUM risk (0.4-0.7)        ({len(medium_risk)} pts)
//
//   Ensemble: RF({W_RF}) + XGB({W_XGB}) + LGB({W_LGB}), threshold={THRESHOLD}
// =============================================================

var fireCoords = {jsonmod.dumps([[round(c[0],5),round(c[1],5)] for c in fire_coords])};
var nofireCoords = {jsonmod.dumps([[round(c[0],5),round(c[1],5)] for c in nofire_coords])};
var highRiskCoords = {jsonmod.dumps([[round(c[0],5),round(c[1],5)] for c in high_risk])};
var medRiskCoords = {jsonmod.dumps([[round(c[0],5),round(c[1],5)] for c in medium_risk])};

var ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017");
var westernGhats = ecoregions.filter(ee.Filter.or(
  ee.Filter.eq('ECO_NAME', 'South Western Ghats moist deciduous forests'),
  ee.Filter.eq('ECO_NAME', 'North Western Ghats moist deciduous forests'),
  ee.Filter.eq('ECO_NAME', 'South Western Ghats montane rain forests'),
  ee.Filter.eq('ECO_NAME', 'North Western Ghats montane rain forests')
));
Map.centerObject(westernGhats, 7);

function coordsToFC(arr) {{
  return ee.FeatureCollection(arr.map(function(c) {{
    return ee.Feature(ee.Geometry.Point(c[0], c[1]));
  }}));
}}

Map.addLayer(westernGhats, {{color: '228B22'}}, 'Western Ghats Boundary', true);

var fireFC = coordsToFC(fireCoords);
var nofireFC = coordsToFC(nofireCoords);
var highFC = coordsToFC(highRiskCoords);
var medFC = coordsToFC(medRiskCoords);

Map.addLayer(nofireFC, {{color: '00CC00'}}, ' No Fire Predicted ({len(nofire_coords)} pts)', false);
Map.addLayer(medFC, {{color: 'FFD700'}}, ' Medium Risk ({len(medium_risk)} pts)', true);
Map.addLayer(highFC, {{color: 'FF6600'}}, ' High Risk ({len(high_risk)} pts)', true);
Map.addLayer(fireFC, {{color: 'FF0000'}}, ' Predicted FIRE ({len(fire_coords)} pts)', true);

var legend = ui.Panel({{
  style: {{position: 'bottom-left', padding: '8px 12px', backgroundColor: 'rgba(255,255,255,0.9)'}}
}});
legend.add(ui.Label({{value: 'FUTURE Fire Predictions (2026-2031)', style: {{fontWeight: 'bold', fontSize: '13px'}}}}));

var items = [
  {{color: 'FF0000', label: 'Predicted FIRE (' + fireCoords.length + ' pts)'}},
  {{color: 'FF6600', label: 'High Risk 0.7 (' + highRiskCoords.length + ' pts)'}},
  {{color: 'FFD700', label: 'Medium Risk 0.4-0.7 (' + medRiskCoords.length + ' pts)'}},
  {{color: '00CC00', label: 'No Fire (' + nofireCoords.length + ' pts)'}}
];
items.forEach(function(item) {{
  var row = ui.Panel({{layout: ui.Panel.Layout.flow('horizontal'), style: {{margin: '2px 0'}}}});
  row.add(ui.Label({{style: {{backgroundColor: '#'+item.color, padding: '6px', margin: '0 6px 0 0', border: '1px solid #333'}}}}));
  row.add(ui.Label({{value: item.label, style: {{fontSize: '11px'}}}}));
  legend.add(row);
}});
legend.add(ui.Label({{value: 'Ensemble: RF({W_RF})+XGB({W_XGB})+LGB({W_LGB})', style: {{fontSize: '10px', color: '#666'}}}}));
Map.add(legend);

print(' Future Fire Predictions 2026-2031');
print('  Total points: ' + (fireCoords.length + nofireCoords.length));
print('  Fire predicted: ' + fireCoords.length);
print('  High risk: ' + highRiskCoords.length);
'''

gee_path = f'{PROJECT_DIR}/GEE_Future_FirePrediction_Visualization.js'
with open(gee_path, 'w') as f:
    f.write(gee_script)
print(f' Saved: {gee_path}')

print('\n Section 23 complete! Future predictions done.')


Section